Gold Tables:\
 ├── sales_by_state\
 ├── top_products\
 └── sales_trend

In [0]:
#sales by state 

# df_fact=spark.table("main.ecommerce.fact_sales")

# df_fact.createOrReplaceTempView("fact_sales")




1. Sales by State / Region

In [0]:
df_sales=spark.sql("""SELECT
    customer_state,
    SUM(revenue) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders
FROM main.ecommerce.fact_sales
GROUP BY customer_state
ORDER BY total_revenue DESC
LIMIT 10""")

df_sales.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/gold/sales_by_state")

2. Top Products by Revenue

In [0]:
df_products=spark.sql("""SELECT
    product_id,
    product_category_name,
    SUM(revenue) AS total_revenue,
    COUNT(order_item_id) AS total_items_sold
FROM main.ecommerce.fact_sales
GROUP BY product_id, product_category_name
ORDER BY total_revenue DESC
LIMIT 10""")

df_products.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/gold/top_products")


3. Daily Sales Trend

In [0]:
df_daily_sales=spark.sql("""
SELECT
    order_date,
    SUM(revenue) AS daily_revenue,
    COUNT(DISTINCT order_id) AS total_orders
FROM main.ecommerce.fact_sales
GROUP BY order_date
ORDER BY order_date""")

df_daily_sales.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/gold/daily_sales")

4. Monthly Sales Trend

In [0]:
df_monthly_sales=spark.sql("""
SELECT
    DATE_TRUNC('month', order_date) AS month,
    SUM(revenue) AS monthly_revenue,
    COUNT(DISTINCT order_id) AS total_orders
FROM main.ecommerce.fact_sales
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY month""")

df_monthly_sales.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/gold/monthly_sales")


# df_yearly_sales=spark.sql("""
# SELECT
#     DATE_TRUNC('year', order_date) AS year,
#     SUM(revenue) AS yearly_revenue,
#     COUNT(DISTINCT order_id) AS total_orders
# FROM fact_sales
# GROUP BY DATE_TRUNC('year', order_date)
# ORDER BY year""")

# df_yearly_sales.write.format("delta").mode("overwrite").load("/Volumes/main/ecommerce/lakehouse_vol/gold/yearly_sales")

In [0]:
%skip
%sql

SELECT
    customer_state,
    COUNT(DISTINCT order_id) AS total_orders
FROM fact_sales
GROUP BY customer_state
ORDER BY total_orders DESC;

In [0]:
%skip
%sql
SELECT
    order_id,
    SUM(revenue) AS order_value
FROM fact_sales
GROUP BY order_id;

In [0]:
%sql
SELECT
    customer_state,
    SUM(revenue) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders
FROM main.ecommerce.fact_sales
GROUP BY customer_state
ORDER BY total_revenue DESC
LIMIT 10